# Lecture 7 — Class Exercise
## Heatmap & Waterfall: Netflix Catalogue

> **Push to:** `week07/lecture07_exercise.ipynb`

**Rules:**
1. Heatmap: colour scale must match the data type (sequential for counts, diverging for above/below)
2. Waterfall: use green for additions, red for subtractions, blue for totals
3. Insight title tells the setup-conflict-resolution story (or at minimum states the finding)
4. Annotate at least one cell or bar directly

---


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('../data/netflix_catalogue.csv')
print(f"Loaded: {len(df)} titles")
print(df['type'].value_counts())
print(df.head())


Loaded: 3000 titles
type
Movie      1974
TV Show    1026
Name: count, dtype: int64
      type  release_year  added_year             genre        country rating  \
0    Movie          2014        2016  Sci-Fi & Fantasy         France  PG-13   
1    Movie          2010        2014     Documentaries  United States  TV-MA   
2  TV Show          2011        2012     Kids & Family  United States  TV-14   
3    Movie          2016        2018             Anime          India     PG   
4    Movie          2014        2016     Kids & Family         Canada  TV-MA   

   duration  
0       157  
1       127  
2         6  
3       134  
4        77  


In [2]:
print("Genres:", df['genre'].value_counts().head(8))
print("\nCountries:", df['country'].value_counts().head(8))
print("\nRatings:", df['rating'].value_counts())


Genres: genre
Sports                244
Sci-Fi & Fantasy      213
Kids & Family         209
Crime                 206
Drama                 204
Horror                199
Action & Adventure    198
Thrillers             195
Name: count, dtype: int64

Countries: country
United States     932
India             337
United Kingdom    261
Japan             187
France            176
Canada            164
South Korea       151
Mexico            138
Name: count, dtype: int64

Ratings: rating
TV-MA    840
TV-14    733
PG-13    589
R        312
PG       196
TV-PG    128
G         92
TV-Y7     57
TV-G      53
Name: count, dtype: int64


## Task 1 — Heatmap: content by rating and release decade

**What to build:** A heatmap showing the number of titles by **content rating** (y-axis) and **decade** (x-axis).

**Requirements:**
- Create a 'decade' column: `df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'`
- Filter to TV-14, TV-MA, PG-13, R, PG only (most common ratings)
- Sequential colour scale (Blues)
- Values shown in cells (`text_auto=True`)
- Insight title about which rating dominates which decade


In [8]:
# Task 1
# YOUR CODE HERE
import pandas as pd
import plotly.express as px

# Create decade column
df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'

# Keep only the required ratings
ratings_keep = ['TV-14', 'TV-MA', 'PG-13', 'R', 'PG']
heatmap_df = df[df['rating'].isin(ratings_keep)].copy()

# Count titles by rating and decade
heatmap_data = (
    heatmap_df
    .groupby(['rating', 'decade'])
    .size()
    .reset_index(name='count')
)

# Order decades chronologically
decade_order = sorted(heatmap_data['decade'].unique())

# Create matrix for heatmap
heatmap_matrix = heatmap_data.pivot(
    index='rating',
    columns='decade',
    values='count'
).fillna(0)

heatmap_matrix = heatmap_matrix[decade_order]

# Find strongest cell for annotation
max_value = heatmap_matrix.max().max()
max_pos = heatmap_matrix.stack().idxmax()

# Heatmap
fig = px.imshow(
    heatmap_matrix,
    text_auto=True,
    color_continuous_scale='Blues',
    aspect='auto',
    labels=dict(
        x='Release Decade',
        y='Content Rating',
        color='Number of Titles'
    ),
    title='TV-MA dominates recent decades while PG and PG-13 were more prominent in earlier releases'
)

# Annotation (required)
fig.add_annotation(
    x=max_pos[1],
    y=max_pos[0],
    text=f'Highest count: {int(max_value)}',
	font=dict(size=14, color="black"),
    showarrow=True,
    arrowhead=3,
    ay=-30
)

fig.update_layout(
    title_x=0.5,
    height=600,
    width=900
)

fig.show()

## Task 2 — Waterfall: Movie vs TV Show additions by year

**What to build:** A waterfall chart showing how Netflix's **Movie library** grew year by year (2015-2022).

**Requirements:**
- Filter to Movies only
- Group by `added_year`, count titles per year
- Final bar should be the cumulative total
- Green bars (additions), blue total
- Annotation on the year with the largest single addition
- Insight title naming the growth story


In [32]:
import pandas as pd
import plotly.graph_objects as go

# Movies only
movies = df[
    (df['type'] == 'Movie') &
    (df['added_year'].between(2015, 2022))
].copy()
# Check years available
print(sorted(movies['added_year'].dropna().unique()))

# Count per year
yearly = (
    movies.groupby('added_year')
    .size()
    .reset_index(name='count')
    .sort_values('added_year')
)

print(yearly)

peak = yearly.loc[yearly['count'].idxmax()]

fig = go.Figure()

fig = go.Figure(go.Waterfall(
    measure=["relative"] * len(yearly) + ["total"],
    x=[str(y) for y in yearly["added_year"]] + ["Total"],  # strings!
    y=yearly["count"].tolist() + [0],

    increasing={"marker": {"color": "green"}},
    totals={"marker": {"color": "blue"}},
    connector={"line": {"color": "gray"}},

    text=yearly["count"].tolist() + [yearly["count"].sum()],
    textposition="outside"
))

fig.update_xaxes(
    type="category"
)


fig.update_layout(
    width=1200,
    height=600,
    showlegend=False
)
fig.update_layout(
    title={
        "text": f"Netflix's movie catalogue grew steadily, with its largest yearly increase occurring in {int(peak['added_year'])}",
        "x": 0.5,
        "xanchor": "center"
    },
    xaxis_title="Year Added",
    yaxis_title="Movies Added",
    width=1200,
    height=650,
    showlegend=False,
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(size=13),
    margin=dict(t=100)
)

# Better gridlines
fig.update_yaxes(
    showgrid=True,
    gridcolor="lightgray"
)

# Ensure categorical years
fig.update_xaxes(
    type="category"
)

fig.show()

[np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
   added_year  count
0        2015     71
1        2016     93
2        2017     77
3        2018     79
4        2019     93
5        2020     81
6        2021     83
7        2022     82
